# Lesson 02 - חקר מסגרת הסוכן של Microsoft

ה-**Microsoft Agent Framework (MAF)** הוא מסגרת מאוחדת לבניית סוכני בינה מלאכותית. היא מספקת ארכיטקטורה נקייה והרכבתית עם ארבעת אבני הבניין המרכזיות:

- **לקוח** – מתחבר אל נקודת קצה של מודל בינה מלאכותית ומנהל תקשורת
- **סוכן** – עוטף לקוח עם הוראות והגדרות כלים
- **כלים** – מרחיבים את יכולות הסוכן עם פונקציות מותאמות שהמודל יכול לקרוא להן
- **מפגש** – שומר היסטוריית שיחות לאינטראקציות מרובות סבבים

בשיעור זה, נבנה **סוכן הזמנת נסיעות** שבודק זמינות יעד באמצעות המושגים הללו.


## התקנה


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## הבנת ארכיטקטורת מסגרת הסוכן

מסגרת הסוכן של מיקרוסופט פועלת על פי ארכיטקטורה בשכבות:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **לקוח** – ספק `AzureAIProjectAgentProvider` מתחבר לפריסת Azure OpenAI. הוא מנהל אימות, עיצוב בקשות וניתוח תגובות.
2. **סוכן** – נוצר מהלקוח דרך `provider.create_agent()`, הסוכן משלב גישה למודל עם הוראות (הנחיית מערכת) וכלים.
3. **כלים** – פונקציות פייתון המעוטרות ב-`@tool` שהסוכן יכול להפעיל לביצוע פעולות או לשליפת נתונים.
4. **מושב** – אובייקט `AgentSession` (נוצר דרך `agent.create_session()`) ששומר היסטוריית שיחה, מאפשר דיאלוג רב-סיבובי שבו הסוכן זוכר את ההקשר הקודם.

בואו נבנה כל שכבה שלב אחר שלב.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## הוספת כלים עם המגדיר @tool

כלים מאפשרים ל־agents לבצע פעולות מעבר ליצירת טקסט. המגדיר `@tool` ממיר פונקציית Python רגילה למשהו שה־agent יכול לקרוא לו.

נקודות מפתח:
- השתמש ב־`Annotated[type, "description"]` כדי שהמודל יבין כל פרמטר.
- מחרוזת התיעוד הופכת לתיאור הכלי שהמודל רואה.
- `approval_mode="never_require"` אומר שהכלי רץ אוטומטית ללא אישור משתמש.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## יצירת סוכן עם כלים

כעת אנו משלבים את הלקוח, ההוראות והכלים לתוך סוכן. ה`instructions` משמשות כהנחיית המערכת — הן מגדירות את הפרסונה וההתנהגות של הסוכן.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## שיחות מרובות סבבים עם סשנים

`AgentSession` (נוצר דרך `agent.create_session()`) עוקב אחרי כל ההודעות בשיחה. על ידי העברת אותו סשן לכל קריאה ל-`agent.run()`, הסוכן יכול לגשת לכל היסטוריית השיחה ולפנות חזרה להודעות קודמות.

אנחנו מעבירים `tools=[check_destination_availability]` כדי שהסוכן יוכל לקרוא למבקר הזמינות שלנו בכל סבב.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## סיכום

בשיעור זה חקרתם את ארבעת העמודים של Microsoft Agent Framework:

| מושג | מה למדתם |
|---------|------------------|
| **לקוח** | `AzureAIProjectAgentProvider` מתחבר ל-Azure OpenAI עם אימות מבוסס אישורים |
| **סוכן** | `provider.create_agent()` מאגד חיבור למודל עם הוראות ושם |
| **כלים** | הקישוט `@tool` חושף פונקציות פייתון שהסוכן יכול לקרוא להן |
| **מפגש** | `agent.create_session()` שומר היסטוריית שיחות על פני פניות מרובות |

יחידות הבנייה הללו משתלבות יחד ליצירת סוכנים שיכולים לנהל שיחות טבעיות, לקרוא לפונקציות חיצוניות ולשמור על הקשר — הבסיס לתבניות סוכנות מתקדמות יותר בשיעורים הבאים.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:  
מסמך זה תורגם באמצעות שירות תרגום מבוסס בינה מלאכותית [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, אנא היו מודעים לכך שתרגומים אוטומטיים עלולים להכיל שגיאות או אי דיוקים. יש להתייחס למסמך המקורי בשפתו המקורית כמקור מוסמך. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא נושאים באחריות לכל אי הבנה או פרשנות שגויה הנובעת משימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
